# Notebook 06 — Whisper Basics

Phase 3 opens with the audio side of multimodal. We'll use **Whisper** (S3 §6.3), the canonical open-source STT model, via the production wrapper everyone actually deploys: `faster-whisper`.

## The mental model (S3 §6.1)

A modern voice pipeline looks like this:

```
audio → VAD → STT → LLM → TTS → audio
```

STT is where the audio modality gets converted into the text tokens the LLM already knows how to handle — directly analogous to how a vision encoder converts an image into visual tokens. We're going to focus on this step today.

## Why faster-whisper, not openai-whisper

OpenAI's reference [Whisper](https://github.com/openai/whisper) is slow on CPU. **Never deploy it.** `faster-whisper` (CTranslate2 backend, int8 quantized) gives 4–10× speedup with near-identical quality, and runs comfortably on a MacBook CPU. It's the production default in 2025-2026.

Other variants worth knowing (S3 §6.3):
- **WhisperX** — adds word-level alignment + speaker diarization. We'll use this in nb07.
- **Distil-Whisper** — distilled, ~6× faster again, ~1% WER worse.

## What this notebook covers

1. Generate three audio samples (English / Chinese / mixed) using OpenAI TTS, so the notebook is self-contained.
2. Load `large-v3` via faster-whisper.
3. Transcribe each sample. Look at `language`, `language_probability`, segment-level structure, and word-level timestamps.
4. **Tease the failure mode** that motivates nb07: what does Whisper do with code-switched speech?

## 1. Generate sample audio with OpenAI TTS

We use TTS so the lab is reproducible (your laptop's mic isn't). Cached to `data/audio_samples/` after first generation.

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(dotenv_path="../.env")
client = OpenAI()

AUDIO_DIR = Path("../data/audio_samples")
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

SAMPLES = {
    "en": "Welcome to the multimodal lab. Today we'll explore how Whisper handles different languages.",
    "zh": "欢迎来到多模态实验室。今天我们来看看 Whisper 是怎么处理不同语言的。",
    "mixed": "明天的 standup 我们 review 一下 Q3 的 OKR，然后 sync 一下 backlog。",
}

def make_sample(name: str, text: str, voice: str = "alloy") -> Path:
    path = AUDIO_DIR / f"{name}.mp3"
    if path.exists():
        print(f"  cached: {path.name}")
        return path
    audio = client.audio.speech.create(model="tts-1", voice=voice, input=text)
    audio.stream_to_file(str(path))
    print(f"  generated: {path.name}")
    return path

paths = {name: make_sample(name, text) for name, text in SAMPLES.items()}

In [ ]:
# Inline audio playback so you can listen in the notebook
from IPython.display import Audio, display
for name, path in paths.items():
    print(f"--- {name}: {SAMPLES[name]}")
    display(Audio(str(path)))

## 2. Load faster-whisper

First load downloads ~1.5 GB. `compute_type="int8"` quantizes weights to 8-bit — 4× smaller, near-identical quality. On Apple Silicon, CPU is fine. On NVIDIA, switch to `device="cuda"` and `compute_type="float16"`.

In [ ]:
from faster_whisper import WhisperModel
import time

MODEL_SIZE = "large-v3"   # other choices: tiny, base, small, medium, large-v2

t0 = time.time()
asr = WhisperModel(MODEL_SIZE, device="cpu", compute_type="int8")
print(f"Loaded {MODEL_SIZE} (int8) in {time.time() - t0:.1f}s")

## 3. Transcribe — see the output structure

`transcribe()` returns a generator of segments + a metadata object. Each segment has text, start/end timestamps, and (with `word_timestamps=True`) a list of words with per-word timestamps and probabilities. The metadata includes the **detected language** with confidence.

In [ ]:
def transcribe(audio_path: Path, **kwargs):
    """Returns (segments_list, info). Materializes the generator so we can inspect freely."""
    segments_gen, info = asr.transcribe(
        str(audio_path),
        word_timestamps=True,
        **kwargs,
    )
    return list(segments_gen), info

def show(name, segments, info):
    print(f"\n=== {name} ===")
    print(f"detected language: {info.language}  (p={info.language_probability:.2f})")
    print(f"duration: {info.duration:.1f}s   transcription duration: {info.duration_after_vad:.1f}s")
    for s in segments:
        print(f"  [{s.start:5.2f} → {s.end:5.2f}]  {s.text.strip()}")
    if segments and segments[0].words:
        print("  word-level (first segment):")
        for w in segments[0].words[:8]:
            print(f"    [{w.start:5.2f} → {w.end:5.2f}]  '{w.word}'  p={w.probability:.2f}")

for name, path in paths.items():
    seg, info = transcribe(path)
    show(name, seg, info)

Two things to notice in the output:

1. **`info.language` is a single label per file.** Whisper's language ID looks at the first 30 seconds, picks one language, and commits. There's no per-segment language identification by default — that becomes a problem fast for code-switched speech.
2. **Word-level probabilities are usable signal.** A word with probability < 0.5 is often hallucinated or mis-segmented. Useful as a confidence gate, mirroring the `confidence` field we used in nb04 for vision.

## 4. Look hard at the mixed sample

What did Whisper do with *"明天的 standup 我们 review 一下 Q3 的 OKR"*? Likely one of:

- Detected the file as `zh`, transcribed everything in Chinese characters — the English tech terms got phonetic-Chinese substitutions ("standup" → "站等" or just dropped).
- Detected `en`, kept the English words and approximated the Chinese.
- Detected `zh` and *silently translated* the English bits to Chinese rather than transcribing them.

Run the cell below to see what your run did:

In [ ]:
seg, info = transcribe(paths["mixed"])
print("Spoken text was:")
print(f"  {SAMPLES['mixed']}\n")
print(f"Whisper detected language: {info.language} (p={info.language_probability:.2f})")
print("Transcription:")
for s in seg:
    print(f"  {s.text.strip()}")

Compare the spoken vs the transcribed text character-by-character. Did the words `standup`, `review`, `Q3`, `OKR`, `sync`, `backlog` survive?

## 5. The `task` foot-gun

Whisper has two modes:
- `task="transcribe"` (default) — keep the language as spoken
- `task="translate"` — translate to English

Ever seen a Chinese audio come back as English text from a system that *should* have transcribed it? That's somebody passing `task="translate"` upstream and forgetting. Watch:

In [ ]:
seg, info = transcribe(paths["zh"], task="translate")
print(f"Spoken: {SAMPLES['zh']}")
print(f"task='translate' →  {' '.join(s.text.strip() for s in seg)}")

**Always pin `task="transcribe"` explicitly** when you want a transcript. Translation should be a separate, deliberate step downstream.

## What we learned

- `faster-whisper large-v3` runs on a CPU with int8 quantization, multilingual out of the box (99 languages).
- It returns segment-level + word-level timestamps with per-word probability — useful for confidence gating.
- Language identification is **per-file, not per-segment**. Code-switching breaks this assumption silently.
- `task="translate"` will silently rewrite your output language — pin `transcribe` explicitly.

**Next:** [Notebook 07 — Code-Switching Deep Dive](07_codeswitching.ipynb). Five carefully-constructed mixed samples, four mitigation strategies, real WER numbers.